# FinSight RAG — 1 of 4: Ingestor

**Purpose:** Fetch 10-K / 10-Q filings from SEC EDGAR and convert them into
LangChain `Document` objects — one per page/section — ready for chunking.

**Input:** A stock ticker (e.g. `'V'` for Visa) and form type (`'10-K'`)  
**Output:** `docs` — a list of `Document` objects saved to `CACHE_DIR` on Drive  
**Next notebook:** `chunker_final.ipynb`

---

### Pipeline position
```
[1. Ingestor] → 2. Chunker → 3. Retriever → 4. Chain
SEC EDGAR PDF/HTM → LangChain Documents
```

### Key design decisions
| Decision | Reason |
|----------|--------|
| Cache as `.bin` not `.pdf` | SEC filings are HTML, not PDF — format-neutral extension avoids parser mismatch |
| `index.json` API to find documents | Machine-readable file list with sizes; regex scraping is brittle |
| Size filter `>50KB` | Skips stub exhibits and cover pages; the main report body is always largest |
| `filing_date` / `accession_number` | Exact key names returned by `get_filing_urls` — must match everywhere |
| Cache validity `>100KB` | Anything smaller is a failed/partial download and should be re-fetched |

## Cell 1 — Install dependencies

In [ ]:
# All libraries needed for this notebook.
# beautifulsoup4 handles HTML filings (most modern SEC 10-Ks are HTML, not PDF).
!pip install pdfplumber langchain-core tenacity python-dotenv pypdf beautifulsoup4 -q
print('✅ Dependencies installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 25.6 MB/s eta 0:00:00
✅ Dependencies installed


## Cell 2 — Imports & logging

Using `logging` instead of `print` for internal function output means you can
control verbosity without changing the functions themselves. Set `LOG_LEVEL=WARNING` in production to silence debug output.

In [ ]:
import os, io, re, time, logging, requests, pdfplumber
from pathlib import Path
from bs4 import BeautifulSoup

from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
from langchain_core.documents import Document
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s',
)
logger = logging.getLogger(__name__)

print('✅ Imports done')

✅ Imports done


### Imports & Logging Setup

This cell initializes the libraries and logging configuration used across the ingestion pipeline.

#### Key Components

- **requests**
  - Handles SEC API communication.

- **pdfplumber**
  - Parses PDF text and tables.

- **BeautifulSoup**
  - Extracts readable text from HTML filings.

- **Document**
  - Standard LangChain schema for RAG workflows.

- **logging**
  - Adds centralized debugging and observability.

#### Production Best Practices

- Group imports logically.
- Configure logging early.
- Use environment variables instead of hardcoded secrets.
- Use module-level loggers for maintainability.

## Cell 3 — Mount Google Drive & set paths

All downloaded filings are cached to Drive so they survive Colab session resets.
Re-runs of this notebook use the cache and make zero network calls to SEC.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/finsight-rag-phase1'
CACHE_DIR  = f'{DRIVE_BASE}/data/filings'   # renamed from /pdfs — we cache HTM too

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
print(f'✅ Drive mounted')
print(f'   Filings will cache to: {CACHE_DIR}')

Mounted at /content/drive
✅ Drive mounted
   Filings will cache to: /content/drive/MyDrive/finsight-rag-phase1/data/filings


## Cell 4 — Constants

**`TICKER_TO_CIK`** maps stock tickers to SEC CIK numbers (Central Index Key —
the unique identifier SEC uses for every public company). To add a new company,
find its CIK at [sec.gov/cgi-bin/browse-edgar](https://www.sec.gov/cgi-bin/browse-edgar)
and add it here.

**`SEC_USER_AGENT`** is required by SEC's fair-use policy. Set it in your `.env`
file as `SEC_USER_AGENT=FinSight YourName yourname@email.com`. Without it,
you will receive `403 Forbidden` errors.

In [ ]:
EDGAR_SUBMISSIONS_URL = 'https://data.sec.gov/submissions/CIK{cik:010d}.json'

# SEC requires a descriptive User-Agent — change to your name/email if you hit 403s
HEADERS = {
    'User-Agent': os.getenv('SEC_USER_AGENT', 'FinSight-RAG dev@example.com'),
}

# Add more tickers here as you expand the project
TICKER_TO_CIK: dict[str, int] = {
    'V':    1403161,   # Visa
    'MA':   1141391,   # Mastercard
    'PYPL': 1633917,   # PayPal
    'SQ':   1512673,   # Block (Square)
}

print('✅ Constants set')
print(f'   Tickers available: {list(TICKER_TO_CIK.keys())}')

✅ Constants set
   Tickers available: ['V', 'MA', 'PYPL', 'SQ']


## Cell 5 — HTTP helper with retry & rate limiting

`tenacity` provides the `@retry` decorator. The retry policy here means:
- Wait 2s after failure 1, 4s after failure 2, 8s after failure 3 (exponential backoff)
- Give up after 4 attempts and re-raise the original exception
- Only retry on network errors — not on 4xx client errors like 403

The `time.sleep(0.15)` before every request keeps throughput at ~6 req/s.
SEC's guidelines ask for ≤10 req/s — violating this gets your IP rate-limited.

In [ ]:
@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    retry=retry_if_exception_type(requests.exceptions.RequestException),
    reraise=True,
)
def _get(url: str) -> requests.Response:
    """
    Resilient HTTP GET with:
    - 0.15s polite delay (SEC rate limit: <=10 req/s)
    - 4 retries with exponential backoff (2s → 4s → 8s → 16s)
    - 30s timeout so a hanging request doesn't freeze the notebook
    """
    time.sleep(0.15)
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp

print('✅ _get() defined')

✅ _get() defined


## Cell 6 — `get_filing_urls()`

Queries the SEC EDGAR submissions API to find recent filings for a company.

**Important:** The dict returned by this function uses exactly these key names:
- `filing_date` — not `filed_date`
- `accession_number` — not `accession`

Every downstream function that reads these dicts must use these exact names.
A mismatch causes `KeyError` at runtime.

The sanity-check print at the bottom verifies the keys every time this cell runs.

In [ ]:
def get_filing_urls(
    ticker: str,
    form_type: str = '10-K',
    max_filings: int = 3,
) -> list[dict]:
    """
    Fetch filing metadata from SEC EDGAR submissions API.

    Returns a list of dicts — each dict has EXACTLY these keys:
        ticker           str   e.g. 'V'
        form_type        str   e.g. '10-K'
        filing_date      str   e.g. '2025-11-06'   <-- NOT 'filed_date'
        accession_number str   e.g. '0001403161-25-000089'  <-- NOT 'accession'
        index_url        str   full URL to the filing index page
    """
    cik = TICKER_TO_CIK.get(ticker.upper())
    if cik is None:
        raise ValueError(f"Unknown ticker '{ticker}'. Add it to TICKER_TO_CIK.")

    logger.info('Fetching submissions for %s (CIK %d)...', ticker, cik)
    data   = _get(EDGAR_SUBMISSIONS_URL.format(cik=cik)).json()
    recent = data.get('filings', {}).get('recent', {})

    forms        = recent.get('form', [])
    accessions   = recent.get('accessionNumber', [])
    filing_dates = recent.get('filingDate', [])

    filings = []
    for form, accession, filing_date in zip(forms, accessions, filing_dates):
        if form != form_type:
            continue

        acc_clean = accession.replace('-', '')
        index_url = (
            f'https://www.sec.gov/Archives/edgar/data/{cik}/'
            f'{acc_clean}/{accession}-index.htm'
        )
        filings.append({
            'ticker':           ticker,
            'form_type':        form,
            'filing_date':      filing_date,      # exact key name used everywhere below
            'accession_number': accession,        # exact key name used everywhere below
            'index_url':        index_url,
        })
        if len(filings) >= max_filings:
            break

    logger.info('Found %d %s filings for %s', len(filings), form_type, ticker)
    return filings

print('✅ get_filing_urls() defined')

# Quick sanity check — print the keys so you can verify them
test = get_filing_urls('V', '10-K', max_filings=1)
print(f'\nSanity check — keys returned: {list(test[0].keys())}')
print(f'Most recent filing: {test[0]["filing_date"]} — {test[0]["accession_number"]}')

✅ get_filing_urls() defined

Sanity check — keys returned: ['ticker', 'form_type', 'filing_date', 'accession_number', 'index_url']
Most recent filing: 2025-11-06 — 0001403161-25-000089


## Cell 7 — `_extract_filing_url()`

SEC filings are **packages of files**, not single documents. The index page
lists exhibits, XBRL data, images, and the actual report body — all as separate files.

This function uses SEC's `index.json` API (machine-readable, includes file sizes)
to identify the main report. Strategy:
1. Filter to `.htm` and `.pdf` files only
2. Exclude XBRL viewer pages (filenames starting with `R`)
3. Exclude tiny files under 50KB (cover pages, stub exhibits)
4. Pick the **largest** remaining file — the main report body is always biggest

In [ ]:
def _extract_filing_url(index_url: str) -> str | None:
    """
    Find the primary filing document using SEC's index.json API.

    Strategy:
    1. Convert the index.htm URL to index.json (machine-readable file list)
    2. Filter to .htm/.pdf files that are >50KB (rules out stubs)
    3. Pick the largest — the main 10-K body is always the biggest file

    Why index.json and not regex on the HTML index page?
    - index.json includes file sizes — essential for picking the right document
    - HTML scraping is brittle; JSON is a stable API
    """
    try:
        # Extract CIK and cleaned accession from the index URL
        cik_match = re.search(r'/edgar/data/(\d+)/', index_url)
        acc_match = re.search(r'/(\d{18})/', index_url.replace('-', ''))
        if not cik_match or not acc_match:
            logger.warning('Could not parse CIK/accession from URL: %s', index_url)
            return None

        cik       = cik_match.group(1)
        acc_clean = acc_match.group(1)

        # Fetch the machine-readable file index
        json_url = f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc_clean}/index.json'
        data  = _get(json_url).json()
        files = data.get('directory', {}).get('item', [])

        # Log what's available (skip images/XBRL for readability)
        readable = [f['name'] for f in files
                    if not f['name'].endswith(('.gif', '.jpg', '.xml', '.xsd', '.zip'))]
        print(f'  Filing contains: {readable}')

        # Filter: must be .htm or .pdf, >50KB, not an XBRL viewer (starts with R)
        candidates = [
            f for f in files
            if (
                f['name'].endswith(('.htm', '.pdf'))
                and not f['name'].startswith('R')
                and 'index' not in f['name'].lower()
                and int(f.get('size', 0)) > 50_000
            )
        ]

        if not candidates:
            logger.warning('No suitable document found in filing index')
            return None

        # Pick the largest — main report body is always biggest
        best = max(candidates, key=lambda f: int(f.get('size', 0)))
        url  = f'https://www.sec.gov/Archives/edgar/data/{cik}/{acc_clean}/{best["name"]}'

        print(f'  → Selected: {best["name"]} ({int(best.get("size", 0)) // 1000} KB)')
        return url

    except Exception as exc:
        logger.warning('_extract_filing_url failed: %s', exc)
        return None

print('✅ _extract_filing_url() defined')

✅ _extract_filing_url() defined


## Cell 8 — `_bytes_to_documents()`

Converts raw filing bytes into LangChain `Document` objects.

**Format detection:** The function reads the first 500 bytes to detect whether
the content is HTML or PDF — it never trusts the file extension. This avoids
the bug where an `.htm` file saved with a `.pdf` extension crashes PDF parsers.

**HTML path** (most modern SEC filings):
BeautifulSoup strips scripts/styles, extracts plain text, then splits into
3000-character sections that mimic page structure.

**PDF path** (older filings):
pdfplumber extracts text page by page, with pypdf as a fallback.

In [ ]:
def _bytes_to_documents(raw_bytes: bytes, metadata: dict) -> list[Document]:
    """
    Convert raw filing bytes into LangChain Documents.
    Auto-detects HTML vs PDF from byte content — never assumes format from extension.

    HTML path (most modern SEC filings):
      BeautifulSoup → strip scripts/styles → extract text → split into 3000-char sections

    PDF path (older filings):
      pdfplumber → one Document per page → pypdf as fallback

    Every Document carries metadata: ticker, form_type, filing_date, source, page/section
    This metadata is what powers cited answers in the RAG chain.
    """
    docs = []

    # Detect format from actual content, not filename
    sample  = raw_bytes[:500].decode('utf-8', errors='ignore').lower()
    is_html = any(marker in sample for marker in ['<html', '<!doctype', '<document'])

    # ── HTML filing path ──────────────────────────────────────────────────
    if is_html:
        print('  Format: HTML filing — parsing with BeautifulSoup')
        try:
            soup = BeautifulSoup(raw_bytes, 'html.parser')

            # Strip non-content tags
            for tag in soup(['script', 'style', 'head', 'nav', 'footer']):
                tag.decompose()

            full_text = soup.get_text(separator='\n')

            # Collapse 3+ blank lines to 2 (SEC filings have lots of whitespace)
            full_text = re.sub(r'\n{3,}', '\n\n', full_text).strip()

            # Split into ~3000-char sections to mimic page structure
            # This size balances context richness vs retrieval precision
            section_size = 3000
            sections = [
                full_text[i : i + section_size]
                for i in range(0, len(full_text), section_size)
            ]

            for section_num, text in enumerate(sections, start=1):
                # Skip near-empty sections (whitespace, page headers)
                if len(text.strip()) > 100:
                    docs.append(Document(
                        page_content=text,
                        metadata={**metadata, 'page': section_num},
                    ))

            print(f'  ✅ HTML → {len(docs)} sections extracted')
            return docs

        except Exception as exc:
            logger.warning('HTML parsing failed: %s — trying PDF path', exc)

    # ── PDF filing path ───────────────────────────────────────────────────
    print('  Format: PDF — parsing with pdfplumber')
    try:
        with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ''
                # Append tables as pipe-delimited rows
                for table in page.extract_tables():
                    rows = [' | '.join(str(c or '') for c in row) for row in table]
                    text += '\n' + '\n'.join(rows)
                if text.strip():
                    docs.append(Document(
                        page_content=text,
                        metadata={**metadata, 'page': page_num},
                    ))
        print(f'  ✅ PDF → {len(docs)} pages extracted')

    except Exception as exc:
        logger.warning('pdfplumber failed (%s) — trying pypdf fallback', exc)
        try:
            from pypdf import PdfReader
            reader = PdfReader(io.BytesIO(raw_bytes))
            for page_num, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ''
                if text.strip():
                    docs.append(Document(
                        page_content=text,
                        metadata={**metadata, 'page': page_num},
                    ))
            print(f'  ✅ pypdf fallback → {len(docs)} pages')
        except Exception as exc2:
            logger.error('All parsers failed: %s', exc2)

    return docs

print('✅ _bytes_to_documents() defined')

✅ _bytes_to_documents() defined


## Cell 9 — `ingest_ticker()` — main entry point

Orchestrates the full pipeline: fetch metadata → find document URL → download
(or load from cache) → parse into Documents.

**Cache behaviour:** Files are saved to Drive as `.bin` (format-neutral).
On re-run, if a `.bin` file exists and is >100KB, it is used directly with
no network calls. Files under 100KB are treated as failed downloads and re-fetched.

In [ ]:
def ingest_ticker(
    ticker: str,
    form_type: str = '10-K',
    max_filings: int = 1,
    cache_dir: str = CACHE_DIR,
) -> list[Document]:
    """
    Main entry point. Fetches, caches, and parses SEC filings.

    Cache note:
      Files are saved as .bin (format-neutral) so the parser always
      detects format from byte content — never from the file extension.
      This prevents the bug where HTML saved as .pdf crashes PDF parsers.

      Cache is only used if file is >100KB — smaller files are stubs or
      failed downloads and should be re-fetched.
    """
    Path(cache_dir).mkdir(parents=True, exist_ok=True)
    all_docs = []

    for filing in get_filing_urls(ticker, form_type, max_filings):

        # .bin extension = format-neutral (works for HTML and PDF)
        cache_path = (
            Path(cache_dir)
            / f"{ticker}_{filing['form_type']}_{filing['filing_date']}.bin"
        )

        # Only use cache if file is a real download (>100KB)
        if cache_path.exists() and cache_path.stat().st_size > 100_000:
            print(f"  Cache hit: {cache_path.name} ({cache_path.stat().st_size // 1000} KB)")
            raw_bytes = cache_path.read_bytes()
        else:
            print(f"  Downloading {filing['filing_date']} {filing['form_type']} for {ticker}...")
            doc_url = _extract_filing_url(filing['index_url'])

            if doc_url is None:
                print(f"  ⚠️  No document found for {filing['accession_number']}, skipping.")
                continue

            raw_bytes = _get(doc_url).content
            cache_path.write_bytes(raw_bytes)
            print(f"  ✅ Saved: {cache_path.name} ({len(raw_bytes) // 1000} KB)")

        # Metadata persists through chunking all the way to the cited answer
        meta = {
            'ticker':      ticker,
            'form_type':   filing['form_type'],
            'filing_date': filing['filing_date'],   # filing_date — consistent with get_filing_urls
            'source':      filing['index_url'],
        }

        docs = _bytes_to_documents(raw_bytes, meta)
        all_docs.extend(docs)

    logger.info('Total documents for %s: %d', ticker, len(all_docs))
    return all_docs

print('✅ ingest_ticker() defined')

✅ ingest_ticker() defined


## Cell 10 — Run the ingestor

First run downloads from SEC (~30–60 seconds, ~1–5 MB).  
Subsequent runs load from Drive cache (~5 seconds, zero network calls).

In [ ]:
print('Starting ingestion for Visa 10-K...\n')

docs = ingest_ticker(
    ticker='V',
    form_type='10-K',
    max_filings=1,
    cache_dir=CACHE_DIR,
)

print(f'\n{"✅" if len(docs) > 50 else "⚠️ "} Done — {len(docs)} sections extracted')

Starting ingestion for Visa 10-K...

  Filing contains: ['0001403161-25-000089-index-headers.html', '0001403161-25-000089-index.html', '0001403161-25-000089.txt', 'MetaLinks.json', 'R1.htm', 'R10.htm', 'R100.htm', 'R101.htm', 'R102.htm', 'R103.htm', 'R104.htm', 'R105.htm', 'R11.htm', 'R12.htm', 'R13.htm', 'R14.htm', 'R15.htm', 'R16.htm', 'R17.htm', 'R18.htm', 'R19.htm', 'R2.htm', 'R20.htm', 'R21.htm', 'R22.htm', 'R23.htm', 'R24.htm', 'R25.htm', 'R26.htm', 'R27.htm', 'R28.htm', 'R29.htm', 'R3.htm', 'R30.htm', 'R31.htm', 'R32.htm', 'R33.htm', 'R34.htm', 'R35.htm', 'R36.htm', 'R37.htm', 'R38.htm', 'R39.htm', 'R4.htm', 'R40.htm', 'R41.htm', 'R42.htm', 'R43.htm', 'R44.htm', 'R45.htm', 'R46.htm', 'R47.htm', 'R48.htm', 'R49.htm', 'R5.htm', 'R50.htm', 'R51.htm', 'R52.htm', 'R53.htm', 'R54.htm', 'R55.htm', 'R56.htm', 'R57.htm', 'R58.htm', 'R59.htm', 'R6.htm', 'R60.htm', 'R61.htm', 'R62.htm', 'R63.htm', 'R64.htm', 'R65.htm', 'R66.htm', 'R67.htm', 'R68.htm', 'R69.htm', 'R7.htm', 'R70.htm', 'R71.h

## Cell 11 — Health check

Verify `docs` before passing to `chunker_final.ipynb`.
A problem caught here is much easier to fix than one discovered after indexing into ChromaDB.

In [ ]:
# This cell confirms everything is correct before you move to chunker.py

print('══ HEALTH CHECK ═══════════════════════════════════════════')

# 1. Docs count
if len(docs) > 50:
    print(f'✅ {len(docs)} sections extracted — good volume')
elif len(docs) > 0:
    print(f'⚠️  Only {len(docs)} sections — may be a stub document, check cache')
else:
    print('❌ 0 sections — ingestion failed, check errors above')

# 2. Metadata keys
expected_keys = {'ticker', 'form_type', 'filing_date', 'source', 'page'}
actual_keys   = set(docs[0].metadata.keys()) if docs else set()
if expected_keys == actual_keys:
    print(f'✅ Metadata keys correct: {sorted(actual_keys)}')
else:
    missing = expected_keys - actual_keys
    extra   = actual_keys - expected_keys
    print(f'⚠️  Metadata mismatch — missing: {missing}, extra: {extra}')

# 3. Cache saved to Drive
cached = list(Path(CACHE_DIR).glob('*.bin'))
if cached:
    for f in cached:
        print(f'✅ Cached to Drive: {f.name} ({f.stat().st_size // 1000} KB)')
else:
    print('❌ No cached files found in Drive')

# 4. Sample content
print('\n══ SAMPLE SECTION (section 10) ════════════════════════════')
if len(docs) >= 10:
    print(docs[9].page_content[:400])
    print(f'\nMetadata: {docs[9].metadata}')

print('\n══ RESULT ══════════════════════════════════════════════════')
if len(docs) > 50 and expected_keys == actual_keys and cached:
    print('🎉 Ingestor complete — ready to move to chunker.py')
else:
    print('⚠️  Some checks failed — review output above before proceeding')

══ HEALTH CHECK ═══════════════════════════════════════════
✅ 155 sections extracted — good volume
✅ Metadata keys correct: ['filing_date', 'form_type', 'page', 'source', 'ticker']
✅ Cached to Drive: V_10-K_2025-11-06.bin (2909 KB)

══ SAMPLE SECTION (section 10) ════════════════════════════
ormanceSharesMember
2025-09-30
0001403161
v:NonUnitedStatesCustomersMember
2024-10-01
2025-09-30
0001403161
v:NonUnitedStatesCustomersMember
2023-10-01
2024-09-30
0001403161
v:NonUnitedStatesCustomersMember
2022-10-01
2023-09-30
0001403161
us-gaap:OtherAssetsMember
2024-09-30
0001403161
us-gaap:ForeignCountryMember
2025-09-30
0001403161
v:UncoveredLitigationMember
2024-10-01
2025-09-30
0001403161


Metadata: {'ticker': 'V', 'form_type': '10-K', 'filing_date': '2025-11-06', 'source': 'https://www.sec.gov/Archives/edgar/data/1403161/000140316125000089/0001403161-25-000089-index.htm', 'page': 10}

══ RESULT ══════════════════════════════════════════════════
🎉 Ingestor complete — ready to move to chunk